In [ ]:
"""This function cannot be executed in ipynb
cuz python multiprocessing is only allowed to run in __main__ module """

from typing import Callable
import torch
import torch.distributed as dist
import torch.multiprocessing as mp
import os
from rich import print


def _find_available_port() -> int:
    """find available port."""
    import socket
    port = 29500
    while True:
        with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
            if s.connect_ex(('localhost', port)) != 0:
                return port
            port += 1

def start_tp_process(proc_id:int,
                     world_size:int,
                     func: Callable,
                     args: list=None,
                     kwargs: dict=None):
    rank = proc_id+1
    try:
        dist.init_process_group('nccl',
                                rank=rank,
                                world_size = world_size)
        torch.cuda.set_device(rank)
        with torch.inference_mode():
            args = args or tuple()
            kwargs = kwargs or dict()
            func(rank, *args, **kwargs)
    except Exception as e:
        from traceback import print_exc
        print_exc()
        if dist.is_initialized():
            dist.destroy_process_group()
        raise e

os.environ['CUDA_VISIBLE_DEVICES']="0,1,2,3"
world_size = 4
model_path = "/data/zszhangyk/Delius/models/glm-4v-9b"
os.environ.setdefault("MASTER_ADDR","127.0.0.1")
os.environ.setdefault("MASTER_PORT",str(_find_available_port()))
addr = os.environ['MASTER_ADDR']
port = os.environ['MASTER_PORT']

def func(rank, *args, **kwargs):
    print("### func ###")
    print(dist.get_world_size())
    print(dist.get_rank())
    print("rank: ", str(rank))
    print("##### func #####")

def initialize():
    print(f"MASTER_ADDR={addr}; MASTER_PORT={port}")
    mp_context = mp.spawn(
        start_tp_process,
        args=(
            world_size,
            func,
            (model_path,),
            dict(model_config=None,
                cache_config=None,
                backend_config=None,
                world_size=world_size)
        ),
        nprocs=world_size-1,
        join=False,
        daemon=True
    )
    rank = 0
    try:
        dist.init_process_group('nccl',
                                rank=rank,
                                world_size = world_size)
    except Exception as e:
        from traceback import print_exc
        print_exc()
        if dist.is_initialized():
            dist.destroy_process_group()
        raise e
if __name__ == '__main__':
    initialize()